# 06 · Understanding 3D biological images

    **Audience:** first-year biology students  
    **Time:** about 75 minutes  
    **Prerequisites:** Tutorials 01–03; no deep-learning knowledge needed  
    **Data:** generated locally; no downloads or personal/clinical data.

    ## By the end, you can

- read `(depth, height, width)` volume axes and three orthogonal planes
- convert voxel counts into physical volume with anisotropic spacing
- prepare channel and batch dimensions for PyTorch 3D models

    **How to work:** Run cells from top to bottom (Shift+Enter). Inspect every figure, try the **Your turn** prompts, then compare with the supplied self-checks. A GPU is not necessary unless noted. If you restart the runtime, rerun all cells.

## 1 · A microscope Z-stack is a volume
In confocal or light-sheet microscopy, each 2D frame can be an optical section at a different depth. Stacking them gives a 3D array. A **voxel** is a volumetric sample, like a 3D pixel. Real acquisitions often have different physical spacing along Z versus X/Y; treating voxels as isotropic can distort biological measurements.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
Z,Y,X=32,64,64
z,y,x=np.mgrid[:Z,:Y,:X]
spacing_zyx_um=(2.0,0.5,0.5)  # depth, row, column
mask=(((z-16)/7)**2+((y-33)/18)**2+((x-31)/14)**2)<=1
rng=np.random.default_rng(6)
volume=np.clip(0.08+0.85*np.exp(-1.4*(((z-16)/7)**2+((y-33)/18)**2+((x-31)/14)**2)) + rng.normal(0,0.05,(Z,Y,X)),0,1).astype('float32')
print('Volume shape (Z,Y,X):',volume.shape)
assert volume.shape==mask.shape==(32,64,64)

## 2 · The XY, XZ and YZ views
The three planes are orthogonal slices through the same object. Use physical extents (in µm) so that the axes reflect acquisition spacing. Otherwise an ellipsoid may look artificially flattened or stretched.

In [ ]:
sz,sy,sx=spacing_zyx_um
fig,axes=plt.subplots(1,3,figsize=(12,3.5))
axes[0].imshow(volume[Z//2],cmap='gray',extent=[0,X*sx,Y*sy,0]);axes[0].set_title('XY (fixed Z)')
axes[1].imshow(volume[:,Y//2,:],cmap='gray',extent=[0,X*sx,Z*sz,0]);axes[1].set_title('XZ (fixed Y)')
axes[2].imshow(volume[:,:,X//2],cmap='gray',extent=[0,Y*sy,Z*sz,0]);axes[2].set_title('YZ (fixed X)')
for ax in axes:
    ax.set_aspect('equal');ax.set_xlabel('µm');ax.set_ylabel('µm')
plt.tight_layout();plt.show()

## 3 · Explore the stack at different depths
A static montage works in any Jupyter environment. The brightest cross-section should occur near the object's middle, not necessarily at the first or last slice.

In [ ]:
chosen_z=[4,10,16,22,28]
fig,axes=plt.subplots(1,len(chosen_z),figsize=(12,3))
for ax,zi in zip(axes,chosen_z):
    ax.imshow(volume[zi],cmap='gray',vmin=0,vmax=1)
    ax.set_title(f'z={zi}');ax.axis('off')
plt.tight_layout();plt.show()

## 4 · Biological volume is not a voxel count
If voxel spacing is `(2.0, 0.5, 0.5)` µm, one voxel occupies `2.0×0.5×0.5 = 0.5 µm³`. Multiply the number of foreground voxels by this physical voxel volume.

In [ ]:
voxel_volume_um3=float(np.prod(spacing_zyx_um))
object_volume_um3=int(mask.sum())*voxel_volume_um3
print('Number of foreground voxels:',int(mask.sum()))
print('Each voxel occupies:',voxel_volume_um3,'µm³')
print('Estimated object volume:',round(object_volume_um3,1),'µm³')
assert np.isclose(voxel_volume_um3,0.5)

## 5 · Dimensions for a 3D neural network
A NumPy volume often has `(D,H,W)`. PyTorch `Conv3d` expects `(N,C,D,H,W)` where `N` is the number of volumes and `C` is the number of input channels. Extra dimensions are not extra biological spatial axes.

In [ ]:
import torch
volume_tensor=torch.from_numpy(volume)[None,None]  # add N and C axes
mask_tensor=torch.from_numpy(mask.astype('float32'))[None,None]
print('Image:',tuple(volume_tensor.shape),'Mask:',tuple(mask_tensor.shape))
assert tuple(volume_tensor.shape)==(1,1,32,64,64)
assert tuple(mask_tensor.shape)==tuple(volume_tensor.shape)

## Your turn
1. What happens to the physical volume calculation if Z spacing is 1.0 rather than 2.0 µm?
2. Show an XY slice at `z=10` instead of `z=16`.
3. Why would a network trained on a 2D slice sometimes miss information visible in adjacent Z slices?

**Self-check:** The shape alone cannot tell you physical voxel size.

In [ ]:
half_z_spacing=(1.0,0.5,0.5)
assert np.isclose(mask.sum()*np.prod(half_z_spacing),object_volume_um3/2)
print('Spacing check passed.')

## Next
We can now replace `Conv2d` with `Conv3d` and create a tiny volumetric segmentation model. A real acquisition also carries orientation metadata and may require resampling: these topics appear in Tutorial 08.